![](https://cascade.yonsei.ac.kr/workshop/rum26/rum26.jpg)

Hosted at Yonsei University, Seoul, South Korea (=Republic of Korea). A five‑day meeting for the RAMSES community and newcomers: code development, science highlights, tutorials, and collaborations.

 - Dates: Apr 30 (Thu)  2026

 - Venue: Yonsei University, Science Building B102



# **Nways GPU Profiling Workshop (Fortran Version)**

Welcome to the Nways GPU Profiling Workshop! 

As scientists and researchers, our main goal is to accelerate our applications using GPUs without getting lost in computer science jargon. During the previous labs, you worked on the Radial Distribution Function (RDF) code.

In this 2-hour hands-on session, we will not just run the code. We will look under the hood using **NVIDIA Profiling Tools (Nsight Systems & Nsight Compute)** to understand *why* our code is slow, *what* the compiler is doing, and *how* to verify our optimizations.

### **Step 0: Check the Environment**
First, let's verify our NVIDIA GPU, the installation paths of our tools (`ncu`, `nsys`), and the HPC SDK libraries.

In [ ]:
%%bash
echo "=================================================="
echo "                GPU & Driver Information          "
echo "=================================================="
nvidia-smi --query-gpu=gpu_name,compute_cap,driver_version --format=csv

echo -e "\n=================================================="
echo "             Tool Installation Paths              "
echo "=================================================="
echo "Compiler (nvfortran): $(which nvfortran)"
echo "Profiler (nsys):      $(which nsys)"
echo "Profiler (ncu):       $(which ncu)"

echo -e "\n=================================================="
echo "             CUDA Toolkit & Compilers             "
echo "=================================================="
nvfortran --version | head -n 2

echo -e "\n=================================================="
echo "             CUDA Library Information             "
echo "=================================================="
echo "Checking key libraries in /opt/nvidia/hpc_sdk/Linux_x86_64/2024/cuda/lib64:"
ls -l /opt/nvidia/hpc_sdk/Linux_x86_64/2024/cuda/lib64 | grep -E "libcudart|libnvToolsExt" | awk '{print $9}'

--- 
## **Part 1: Compiling and OpenACC Built-in Profiling**

Before we even run the advanced profiler, the compiler can tell us a lot about what it is doing with our GPU. 

### **1.1 Inspecting the Source Code**
We will be working with the file: `openacc/source_code/SOLUTION/rdf_parallel_directive.f90`

Let's look at the core loop of our **Naive Parallel** OpenACC code. In this code, we only added simple `!$acc` directives to parallelize the loop, but we completely ignored memory transfers.

In [ ]:
%%bash
cd openacc/source_code

echo "--- Core Loop of rdf_parallel_directive.f90 ---"
# We use grep -i to catch case-insensitive Fortran pragmas without bash escaping issues
grep -i -A 15 "acc parallel" SOLUTION/rdf_parallel_directive.f90

### **1.2 Compiling with Feedback (`-Minfo`)**
**Important Educational Lesson:** If we don't explicitly define data regions, the compiler might safely attempt to copy the entire array every time the loop is reached, or in some configurations, fail to copy allocatable arrays efficiently. 

To ensure a robust educational baseline and automatically map CPU arrays to GPU memory, we will tell the compiler to use **Unified Memory** by adding the `-gpu=managed` flag. Also, unlike C++, Fortran requires us to explicitly link the NVTX library (`-lnvToolsExt`) to use profiling markers.

In [ ]:
%%bash
cd openacc/source_code/SOLUTION

# In Linux, library linking order matters! We must place NVTX_LINK at the END of the compile command.
NVTX_LINK="-cudalib=nvtx -L/opt/nvidia/hpc_sdk/Linux_x86_64/2024/cuda/lib64 -lnvToolsExt"

echo "Compiling Naive Parallel Version with Unified Memory (-gpu=managed)..."
nvfortran -O3 -w -acc -gpu=cc89,managed -Minfo=accel -o rdf_parallel_f rdf_parallel_directive.f90 ${NVTX_LINK}

**🔍 Check the Output:**
You should see `Generating NVIDIA GPU code`: The compiler successfully created a GPU kernel!

### **1.3 Executing the Code (Normal vs. Profile Mode)**
First, let's execute the code normally. *(Note: We execute the code from the `source_code` directory so it can find the dataset at `../../_common/input/alk.traj.dcd`)*

In [ ]:
%%bash
cd openacc/source_code

echo "Executing normally..."
./SOLUTION/rdf_parallel_f

The code runs and prints the output, but it doesn't tell us how the GPU performed. 

Before using heavy advanced profilers, the NVIDIA HPC SDK provides a fantastic built-in tool. By setting `NVCOMPILER_ACC_TIME=1` before execution, we get an immediate educational breakdown of kernel execution time and data transfer time. Let's try it!

In [ ]:
%%bash
cd openacc/source_code

echo "Executing with NVCOMPILER_ACC_TIME=1..."
NVCOMPILER_ACC_TIME=1 ./SOLUTION/rdf_parallel_f

**💡 What to look for:** At the very end of the execution output, you will now see an `Accelerator Kernel Timing data` table showing exactly how many microseconds were spent on the compute kernel versus Unified Memory page faults (`data copyin`, `data copyout`).

---
## **Part 2: Profiling with Nsight Systems (`nsys`)**

While the built-in time debug is great, **Nsight Systems (`nsys`)** creates a full visual timeline. By using the `--sample=cpu` and `--stats=true` flags, we can also see a `gprof`-like subroutine profile directly in the standard output.

### **2.1 Profile the Naive OpenACC Version**
We will profile the naive version we just compiled and let `nsys` print the full statistics to the screen.

In [ ]:
%%bash
cd openacc/source_code

echo "Profiling Naive Parallel Version (stdout is displayed!)..."
nsys profile \
  --trace=cuda,openacc,osrt,nvtx \
  --sample=cpu \
  --stats=true \
  --force-overwrite=true \
  -o profile_acc_parallel_f \
  ./SOLUTION/rdf_parallel_f

**🔍 Look at the Console Output:** Scroll up through the output of `nsys profile`. You will see tables for `NVTX Range Statistics` (showing the "Pair_Calculation" function), `CUDA Kernel Statistics`, and `CUDA Memory Operation Statistics`. Notice how much time is spent on Memory Operations (HtoD / DtoH)!

### **2.2 Inspecting the Data Optimized Code**
Let's look at `rdf_data_directive.f90`. Notice how we explicitly manage data using `!$acc data copyin(...) copy(...)`. Because we explicitly define the data boundaries here, we no longer need the `-gpu=managed` flag!

In [ ]:
%%bash
cd openacc/source_code

echo "--- Inspecting Data Directives ---"
grep -i -B 2 -A 10 "acc data" SOLUTION/rdf_data_directive.f90

### **2.3 Compiling the Data Optimized Version**

In [ ]:
%%bash
cd openacc/source_code/SOLUTION

NVTX_LINK="-cudalib=nvtx -L/opt/nvidia/hpc_sdk/Linux_x86_64/2024/cuda/lib64 -lnvToolsExt"

echo "Compiling Data Optimized Version (No 'managed' flag needed!)..."
nvfortran -O3 -w -acc -gpu=cc89 -Minfo=accel -o rdf_data_f rdf_data_directive.f90 ${NVTX_LINK}

### **2.4 Profiling the Data Optimized Version**
Let's trace this optimized version. Pay attention to the `CUDA Memory Operation Statistics` table this time!

In [ ]:
%%bash
cd openacc/source_code

echo "Profiling Data Optimized Version..."
nsys profile \
  --trace=cuda,openacc,osrt,nvtx \
  --stats=true \
  --force-overwrite=true \
  -o profile_acc_data_f \
  ./SOLUTION/rdf_data_f

**💡 What changed?** Check the memory operations table in the output above. The overhead is significantly less, and the overall kernel execution time decreased dramatically!

---
## **Part 3: OpenMP Offload - Expanding Parallelism (Collapse)**

OpenMP is another popular model. A common optimization is **Loop Collapsing**. 
If you have two nested loops (e.g., `do i = 1, 100`, `do j = 1, 100`), they create $100 \times 100 = 10,000$ independent work items. Collapsing them allows the GPU to process all 10,000 threads simultaneously across its massive array of cores instead of managing them hierarchically.

### **3.1 Inspecting Basic OpenMP Offload Code**

In [ ]:
%%bash
cd openmp/source_code

echo "--- Basic OpenMP Target Region ---"
grep -i -A 5 "omp target teams distribute" SOLUTION/rdf_offload.f90

### **3.2 Compiling & Profiling Basic OpenMP Offload**

In [ ]:
%%bash
cd openmp/source_code

NVTX_LINK="-cudalib=nvtx -L/opt/nvidia/hpc_sdk/Linux_x86_64/2024/cuda/lib64 -lnvToolsExt"

echo "Compiling Basic OpenMP Offload..."
nvfortran -O3 -w -mp=gpu -gpu=cc89 -o SOLUTION/rdf_omp_offload_f SOLUTION/rdf_offload.f90 ${NVTX_LINK}

echo "Profiling Basic OpenMP Offload..."
nsys profile \
  --trace=cuda,openmp,osrt,nvtx \
  --stats=true \
  --force-overwrite=true \
  -o profile_omp_offload_f \
  ./SOLUTION/rdf_omp_offload_f

### **3.3 Inspecting Collapsed OpenMP Code**
Notice the `collapse(2)` clause added to the pragmas.

In [ ]:
%%bash
cd openmp/source_code

echo "--- Collapsed OpenMP Target Region ---"
grep -i -A 5 "omp target teams distribute" SOLUTION/rdf_offload_collapse.f90

### **3.4 Compiling & Profiling Collapsed OpenMP Offload**

In [ ]:
%%bash
cd openmp/source_code

NVTX_LINK="-cudalib=nvtx -L/opt/nvidia/hpc_sdk/Linux_x86_64/2024/cuda/lib64 -lnvToolsExt"

echo "Compiling Collapsed OpenMP Offload..."
nvfortran -O3 -w -mp=gpu -gpu=cc89 -o SOLUTION/rdf_omp_collapse_f SOLUTION/rdf_offload_collapse.f90 ${NVTX_LINK}

echo "Profiling Collapsed OpenMP Offload..."
nsys profile \
  --trace=cuda,openmp,osrt,nvtx \
  --stats=true \
  --force-overwrite=true \
  -o profile_omp_collapse_f \
  ./SOLUTION/rdf_omp_collapse_f

**💡 What to look for:** Look at the `CUDA Kernel Statistics` table above. Compare it with the Basic OpenMP version. The `collapse` version reduces the calculation time per kernel significantly because the GPU is fed much more parallel work simultaneously!

---
## **Part 4: CUDA Fortran - Memory Strategies**

For those writing CUDA Fortran (`.cuf` or `.f90`) code, memory management is highly explicit. You can:
1. Manually allocate device memory using the `device` attribute and move data.
2. Let the NVIDIA driver handle it automatically using **Unified Virtual Memory (UVM)** via the `managed` attribute.

### **4.1 Inspecting Explicit Device Allocation Code**

In [ ]:
%%bash
cd cuda/source_code

echo "--- Explicit Device Memory Allocation Example (rdf_orig.f90) ---"
grep -i -B 1 -A 5 "allocate(" SOLUTION/rdf_orig.f90 | head -n 10

### **4.2 Compiling & Profiling Explicit Allocation Version**

In [ ]:
%%bash
cd cuda/source_code

NVTX_LINK="-cudalib=nvtx -L/opt/nvidia/hpc_sdk/Linux_x86_64/2024/cuda/lib64 -lnvToolsExt"

echo "Compiling Explicit Allocation Version..."
# We use -cuda and -gpu=cc89 so nvfortran accurately parses any CUDA Fortran extensions in the .f90 file
nvfortran -O3 -w -cuda -gpu=cc89 -o SOLUTION/rdf_explicit_f SOLUTION/rdf_orig.f90 ${NVTX_LINK}

echo "Profiling Explicit Allocation..."
nsys profile \
  --trace=cuda,nvtx \
  --stats=true \
  --force-overwrite=true \
  -o profile_cuda_explicit_f \
  ./SOLUTION/rdf_explicit_f

### **4.3 Inspecting Unified Virtual Memory (`managed`) Code**

In [ ]:
%%bash
cd cuda/source_code

echo "--- Unified Memory Example (rdf_unified_memory.f90) ---"
grep -i -B 1 -A 5 "managed" SOLUTION/rdf_unified_memory.f90 | head -n 10

### **4.4 Compiling & Profiling Unified Memory Version**

In [ ]:
%%bash
cd cuda/source_code

NVTX_LINK="-cudalib=nvtx -L/opt/nvidia/hpc_sdk/Linux_x86_64/2024/cuda/lib64 -lnvToolsExt"

echo "Compiling Unified Memory Version..."
nvfortran -O3 -w -cuda -gpu=cc89 -o SOLUTION/rdf_managed_f SOLUTION/rdf_unified_memory.f90 ${NVTX_LINK}

echo "Profiling Unified Memory..."
nsys profile \
  --trace=cuda,nvtx \
  --stats=true \
  --force-overwrite=true \
  -o profile_cuda_managed_f \
  ./SOLUTION/rdf_managed_f

---
## **Part 5: Hardware Deep Dive with Nsight Compute (`ncu`)**

Nsight Systems (`nsys`) gives us the big picture (the forest). Now we will use Nsight Compute (`ncu`) to look closely at a single kernel (the tree).

`ncu` replays the kernel dozens of times to collect detailed hardware metrics (registers, bandwidth, warps, etc.). Because it is so detailed, profiling an entire application will take too long. Therefore, we use specific flags:
* `--launch-skip 2`: Skip the first 2 kernels (to avoid profiling un-warmed-up cache states).
* `--launch-count 1`: Only profile exactly 1 kernel execution and then let the program run normally.

### **⚠️ Note on Cloud Environments & `ERR_NVGPUCTRPERM`**
Nsight Compute requires deep hardware access. In many Docker containers and Cloud environments (like this workspace), this access is blocked by the hypervisor/host for security reasons unless the container is started with `SYS_ADMIN` capabilities. 
If you see `==ERROR== ERR_NVGPUCTRPERM` below, **do not panic!** It is entirely expected in a secured cloud container. The Jupyter cell is configured to catch this safely so we can learn from it and proceed.

In [ ]:
%%bash
cd openacc/source_code

echo "Running Detailed Hardware Analysis with Nsight Compute..."
# The '|| true' statement at the end prevents Jupyter from crashing if the cloud environment blocks hardware counters.
ncu \
  --launch-skip 2 \
  --launch-count 1 \
  --set full \
  --force-overwrite \
  -o ncu_acc_data_f \
  ./SOLUTION/rdf_data_f || echo -e "\n✅ [Educational Note]: Nsight Compute failed due to ERR_NVGPUCTRPERM. This confirms our cloud container restricts hardware counter access. This is expected, please proceed to Part 6!"


---
## **Part 6: Final Task - Full Profiling and Local UI Analysis**

To conclude our workshop, we will generate a comprehensive profile report containing all possible trace information. This generates an **`.nsys-rep`** file which you will download to your local machine.

Let's compile and trace the fully optimized OpenACC code (`rdf_gang_vector`) with everything turned on.

In [ ]:
%%bash
cd openacc/source_code

NVTX_LINK="-cudalib=nvtx -L/opt/nvidia/hpc_sdk/Linux_x86_64/2024/cuda/lib64 -lnvToolsExt"

echo "Compiling the fully optimized gang_vector version..."
nvfortran -O3 -w -acc -gpu=cc89 -o SOLUTION/rdf_final_f SOLUTION/rdf_gang_vector.f90 ${NVTX_LINK}

echo "Generating comprehensive profile report (.nsys-rep)..."
nsys profile \
  --trace=cuda,openacc,osrt,nvtx \
  --sample=cpu \
  --stats=true \
  --force-overwrite=true \
  -o rdf_final_comprehensive_f \
  ./SOLUTION/rdf_final_f > /dev/null 2>&1
  
echo "File generated: /labs/openacc/source_code/rdf_final_comprehensive_f.nsys-rep"

### **🖥️ Local UI Exercise: Open and Analyze Your Profiles!**

The command line numbers are great, but visualizing the timeline is where the real power of profiling lies.

**⚠️ IMPORTANT: How to Download (Avoid UTF-8 Error)**
Because `.nsys-rep` files are binary databases, simply left-clicking on them in JupyterLab will cause a "not UTF-8 encoded" error. To download them correctly, you must use your browser's native context menu.

**Please follow these exact steps to download:**
1. Hover over the links below.
2. Hold the **Shift** key and **Right-Click** on the link to open your browser's native menu (this overrides the Jupyter menu).
3. Select **"Save Link As..."** (or "Download Linked File") to save it to your local laptop.

*(Note: The files are located relative to the `/labs` directory where this notebook is running)*

* [Download rdf_final_comprehensive_f.nsys-rep](openacc/source_code/rdf_final_comprehensive_f.nsys-rep) (Part 6: Fully Optimized OpenACC)
* [Download profile_acc_parallel_f.nsys-rep](openacc/source_code/profile_acc_parallel_f.nsys-rep) (Part 2.1: Naive Parallel OpenACC)
* [Download profile_cuda_managed_f.nsys-rep](cuda/source_code/profile_cuda_managed_f.nsys-rep) (Part 4.4: CUDA Unified Memory)
* [Download profile_cuda_explicit_f.nsys-rep](cuda/source_code/profile_cuda_explicit_f.nsys-rep) (Part 4.2: Explicit Allocation)

---
**How to view your profiles:**
1. Open **NVIDIA Nsight Systems** on your local machine.
2. Drag and drop the downloaded `.nsys-rep` files into the Nsight Systems window.

**Checklist of what to look for in the UI:**
* 🔍 **NVTX Markers:** Look at the top rows. You will see colored bars labeled "Pair_Calculation". This helps you isolate exactly where your function is running!
* 🔍 **Compare Parallel vs Final:** Notice the overwhelming amount of green "Unified Memory" page faults or memory transfers hiding the compute blocks in early versions. Now open the `rdf_final_comprehensive_f` file—you should see almost entirely pure compute blocks. 
* 🔍 **Unified Memory (from Part 4):** Open `profile_cuda_managed_f.nsys-rep` and look for "Page Faults" occurring *during* the kernel execution. Compare this to `profile_cuda_explicit_f.nsys-rep` where transfers happen explicitly before the kernel starts.

Congratulations on completing the profiling workshop! You now have the skills to compile, trace, measure, and visually inspect GPU performance for any scientific application.